# TCGA-BRCA Minimal Dry-Run Cohort Review

This notebook is review-only. It loads the latest saved minimal dry-run cohort outputs from disk, checks the run-level validation state, reviews join and mismatch signals, and writes review tables for human audit.


## Load the latest saved minimal dry-run cohort build


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')


def read_tsv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False)


repo_root = find_repo_root(Path.cwd())
latest_pointer_path = (
    repo_root
    / '01-data'
    / 'audit'
    / 'tcga-brca'
    / 'cohort'
    / 'tcga_brca_minimal_dry_run_cohort_latest.json'
)
if not latest_pointer_path.exists():
    raise FileNotFoundError(
        f'Latest minimal dry-run cohort pointer not found: {latest_pointer_path}. Run the build script first.'
    )

latest_pointer = json.loads(latest_pointer_path.read_text(encoding='utf-8'))
cohort_path = repo_root / latest_pointer['minimal_dry_run_cohort_tsv']
join_audit_path = repo_root / latest_pointer['minimal_dry_run_join_audit_tsv']
row_counts_path = repo_root / latest_pointer['minimal_dry_run_row_counts_tsv']
exclusions_path = repo_root / latest_pointer['minimal_dry_run_exclusions_tsv']
summary_path = repo_root / latest_pointer['minimal_dry_run_summary_tsv']
run_log_path = repo_root / latest_pointer['run_log_json']
results_root = repo_root / '09-trials' / '01-tcga-only-source-audited' / '05-results'
results_root.mkdir(parents=True, exist_ok=True)

display(pd.DataFrame([latest_pointer]))


,updated_at_utc,dry_run_build_id,blueprint_run_id,ambiguity_resolution_run_id,shortlist_run_id,core_audit_run_id,clinical_parse_run_id,endpoint_crosswalk_run_id,biospecimen_crosswalk_run_id,biospecimen_parse_run_id,...,minimal_dry_run_exclusions_tsv,minimal_dry_run_summary_tsv,run_log_json,cohort_blueprint_latest_json,blueprint_ambiguity_resolution_latest_json,clinical_shortlist_latest_json,endpoint_crosswalk_latest_json,biospecimen_crosswalk_latest_json,clinical_biotab_latest_json,biospecimen_biotab_latest_json
0,2026-04-13T18:59:12Z,20260413T185911Z,20260413T081835Z,20260413T085636Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T042036Z,20260413T073640Z,20260413T070359Z,...,01-data/audit/tcga-brca/cohort/minimal_dry_run...,01-data/audit/tcga-brca/cohort/minimal_dry_run...,01-data/audit/tcga-brca/cohort/minimal_dry_run...,01-data/audit/tcga-brca/cohort/tcga_brca_cohor...,01-data/audit/tcga-brca/cohort/tcga_brca_bluep...,01-data/audit/tcga-brca/variables/tcga_brca_cl...,01-data/audit/tcga-brca/variables/tcga_brca_en...,01-data/audit/tcga-brca/variables/tcga_brca_bi...,01-data/audit/tcga-brca/variables/tcga_brca_cl...,01-data/audit/tcga-brca/variables/tcga_brca_bi...


## Load saved dry-run artifacts


In [2]:
cohort_df = read_tsv(cohort_path)
join_audit_df = read_tsv(join_audit_path)
row_counts_df = read_tsv(row_counts_path)
exclusions_df = read_tsv(exclusions_path)
summary_df = read_tsv(summary_path)
run_log = json.loads(run_log_path.read_text(encoding='utf-8'))

if run_log.get('status') != 'completed':
    raise ValueError(f"Dry-run build is not completed: status={run_log.get('status')}")
if not run_log.get('validation', {}).get('passed', False):
    raise ValueError('Dry-run build validation did not pass. Review the saved run_log.json before continuing.')

print(f"Dry-run build ID: {latest_pointer['dry_run_build_id']}")
print(f"Blueprint run ID: {latest_pointer['blueprint_run_id']}")
print(f"Ambiguity-resolution run ID: {latest_pointer['ambiguity_resolution_run_id']}")
print(f"Cohort TSV: {cohort_path}")
print(f"Join audit TSV: {join_audit_path}")
print(f"Row counts TSV: {row_counts_path}")
print(f"Exclusions TSV: {exclusions_path}")
print(f"Summary TSV: {summary_path}")
print(f"Run log: {run_log_path}")

validation_df = pd.DataFrame([run_log.get('validation', {})])
display(validation_df)


Dry-run build ID: 20260413T185911Z
Blueprint run ID: 20260413T081835Z
Ambiguity-resolution run ID: 20260413T085636Z
Cohort TSV: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\cohort\minimal_dry_run_runs\20260413T185911Z\minimal_dry_run_cohort.tsv
Join audit TSV: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\cohort\minimal_dry_run_runs\20260413T185911Z\minimal_dry_run_join_audit.tsv
Row counts TSV: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\cohort\minimal_dry_run_runs\20260413T185911Z\minimal_dry_run_row_counts.tsv
Exclusions TSV: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\cohort\minimal_dry_run_runs\20260413T185911Z\minimal_dry_run_exclusions.tsv
Summary TSV: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\cohort\minimal_dry_run_runs\20260413T185911Z\minimal_dry_run_summary.tsv
Run log: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\cohort\minimal_dry_run_runs\20260413T185911Z\run_log.json


,passed,blueprint_latest_pointer_found,ambiguity_resolution_latest_pointer_found,clinical_shortlist_latest_pointer_found,endpoint_crosswalk_latest_pointer_found,biospecimen_crosswalk_latest_pointer_found,clinical_biotab_latest_pointer_found,biospecimen_biotab_latest_pointer_found,blueprint_run_log_completed,ambiguity_resolution_run_log_completed,...,cohort_row_ids_are_sequential,join_audit_row_count_positive,row_counts_row_count_positive,exclusions_row_count_positive,summary_row_count_positive,final_cohort_row_count_matches_clinical_patient,summary_contains_readiness_interpretation,summary_contains_mismatch_note,no_prior_run_overwrite,latest_pointer_written_after_success_only
0,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True


## Review row counts, join success, mismatch patterns, and carried ambiguity flags


In [3]:
def parse_json_list(value: str) -> list[str]:
    if value == '':
        return []
    parsed = json.loads(value)
    if not isinstance(parsed, list):
        raise ValueError(f'Expected a JSON list, received: {value}')
    return [str(item) for item in parsed]


summary_review_df = summary_df.sort_values(['summary_section', 'summary_metric']).reset_index(drop=True)
join_review_df = (
    join_audit_df.assign(join_step_order_numeric=pd.to_numeric(join_audit_df['join_step_order'], errors='raise'))
    .sort_values('join_step_order_numeric')
    .drop(columns='join_step_order_numeric')
    .reset_index(drop=True)
)
row_counts_review_df = row_counts_df.sort_values(['count_stage', 'count_metric']).reset_index(drop=True)
exclusions_review_df = exclusions_df.sort_values(
    ['exclusion_category', 'source_table', 'entity_value'],
    ascending=[True, True, True],
).reset_index(drop=True)
cohort_sorted_df = (
    cohort_df.assign(
        provisional_patient_row_id_numeric=pd.to_numeric(
            cohort_df['provisional_patient_row_id'],
            errors='raise',
        )
    )
    .sort_values('provisional_patient_row_id_numeric')
    .drop(columns='provisional_patient_row_id_numeric')
    .reset_index(drop=True)
)

ambiguity_flags_review_df = (
    cohort_sorted_df[
        [
            'dry_run_build_id',
            'provisional_patient_row_id',
            'bcr_patient_barcode',
            'bcr_patient_uuid',
            'ambiguity_note_flags_json',
        ]
    ]
    .assign(ambiguity_note_flag=lambda df: df['ambiguity_note_flags_json'].map(parse_json_list))
    .explode('ambiguity_note_flag')
    .drop(columns='ambiguity_note_flags_json')
    .reset_index(drop=True)
)
join_status_flags_review_df = (
    cohort_sorted_df[
        [
            'dry_run_build_id',
            'provisional_patient_row_id',
            'bcr_patient_barcode',
            'bcr_patient_uuid',
            'join_status_flags_json',
        ]
    ]
    .assign(join_status_flag=lambda df: df['join_status_flags_json'].map(parse_json_list))
    .explode('join_status_flag')
    .drop(columns='join_status_flags_json')
    .reset_index(drop=True)
)

ambiguity_flag_counts_df = (
    ambiguity_flags_review_df.groupby('ambiguity_note_flag', dropna=False)
    .size()
    .reset_index(name='row_count')
    .sort_values(['row_count', 'ambiguity_note_flag'], ascending=[False, True])
    .reset_index(drop=True)
)
join_status_flag_counts_df = (
    join_status_flags_review_df.groupby('join_status_flag', dropna=False)
    .size()
    .reset_index(name='row_count')
    .sort_values(['row_count', 'join_status_flag'], ascending=[False, True])
    .reset_index(drop=True)
)
key_summary_df = summary_review_df.loc[
    summary_review_df['summary_metric'].isin(
        [
            'proposed_patient_universe_row_count',
            'final_dry_run_row_count',
            'matched_followup_candidate_count',
            'matched_followup_row_count',
            'matched_biospecimen_sample_anchor_count',
            'matched_biospecimen_sample_row_count',
            'clinical_patient_vs_source_vs_biospecimen_patient_counts',
            'dry_run_readiness_interpretation',
        ]
    )
].reset_index(drop=True)
mismatch_review_df = exclusions_review_df.loc[
    exclusions_review_df['exclusion_category'].isin(
        [
            'unmatched_source_case_submitter_id',
            'left_unmatched_join_group',
            'unmatched_biospecimen_patient_uuid',
        ]
    )
].reset_index(drop=True)

display(key_summary_df)
display(join_review_df)
display(join_status_flag_counts_df)
display(ambiguity_flag_counts_df)
display(mismatch_review_df)


,dry_run_build_id,summary_section,summary_metric,summary_value,notes
0,20260413T185911Z,join_results,matched_biospecimen_sample_anchor_count,1097,Clinical patient rows with at least one groupe...
1,20260413T185911Z,join_results,matched_biospecimen_sample_row_count,2293,biospecimen_sample source rows grouped into ma...
2,20260413T185911Z,join_results,matched_followup_candidate_count,619,Clinical patient rows with at least one groupe...
3,20260413T185911Z,join_results,matched_followup_row_count,716,Follow-up source rows grouped into matched pat...
4,20260413T185911Z,mismatch_signals,clinical_patient_vs_source_vs_biospecimen_pati...,1097 / 1098 / 1101,This dry run preserves the current count misma...
5,20260413T185911Z,readiness,dry_run_readiness_interpretation,sufficient_for_join_test_not_final_freeze,The saved evidence is sufficient to execute an...
6,20260413T185911Z,row_counts,final_dry_run_row_count,1097,Final dry-run cohort row count.
7,20260413T185911Z,row_counts,proposed_patient_universe_row_count,1097,clinical_patient source-row count used as the ...


,dry_run_build_id,join_step_order,join_step_name,left_table_name,right_table_name,starting_left_row_count,matched_left_row_count,unmatched_left_row_count,matched_right_row_count,right_only_row_count,left_rows_with_multiple_matches,max_right_matches_per_left_row,join_rule_used,ambiguity_carried_forward,notes
0,20260413T185911Z,1,clinical_patient baseline build,patient/case,clinical_patient,1097,1097,0,1097,0,0,1,Use clinical_patient as the provisional patien...,[],Baseline fields come only from cohort_blueprin...
1,20260413T185911Z,2,attach patient-layer endpoint candidates,clinical_patient,clinical_patient,1097,1097,0,1097,0,0,1,Carry patient-table endpoint candidates in pla...,"[""endpoint_overlap_fields_kept_side_by_side""]",Attached patient-layer candidate fields: last_...
2,20260413T185911Z,3,attach follow-up-layer endpoint candidates,clinical_patient,clinical_follow_up_v4_0,1097,619,478,716,0,91,3,Group clinical_follow_up_v4_0 by exact (bcr_pa...,"[""endpoint_overlap_fields_kept_side_by_side""]",Attached follow-up candidate fields as provena...
3,20260413T185911Z,4,attach biospecimen sample anchor evidence,clinical_patient,biospecimen_sample,1097,1097,0,2293,9,1097,4,Group biospecimen_sample by bcr_patient_uuid o...,"[""parallel_patient_identifiers_retained"", ""cas...",Biospecimen sample evidence remains UUID-ancho...


,join_status_flag,row_count
0,biospecimen_sample_matched,1097
1,biospecimen_sample_multirow,1097
2,biospecimen_uuid_only_cross_layer_join,1097
3,followup_matched,619
4,followup_unmatched,478
5,followup_multirow,91


,ambiguity_note_flag,row_count
0,biospecimen_child_layer_expansion_deferred,1097
1,case_count_mismatch_carried_forward_1097_1098_...,1097
2,endpoint_overlap_fields_kept_side_by_side,1097
3,parallel_patient_identifiers_retained,1097
4,sparse_timing_fields_excluded_from_minimal_build,1097
5,treatment_detail_rows_excluded_from_minimal_build,1097


,dry_run_build_id,exclusion_category,scope_level,source_table,entity_id,entity_value,reason,count_value,details
0,20260413T185911Z,left_unmatched_join_group,patient_rows,clinical_follow_up_v4_0,clinical_patient_without_followup_match,exact_uuid_barcode_join,These clinical_patient rows had no matching gr...,478,Count of clinical_patient rows with zero clini...
1,20260413T185911Z,unmatched_biospecimen_patient_uuid,patient_uuid,biospecimen_sample,bcr_patient_uuid,51d991f6-d998-420e-9357-48736c9ea0c1,biospecimen_sample contains this patient UUID ...,2,"Unmatched sample barcodes=[""TCGA-AN-A0FG-01A"",..."
2,20260413T185911Z,unmatched_biospecimen_patient_uuid,patient_uuid,biospecimen_sample,bcr_patient_uuid,57a1604c-60b7-4b30-a75e-f70939532c5c,biospecimen_sample contains this patient UUID ...,3,"Unmatched sample barcodes=[""TCGA-BH-A0B2-01A"",..."
3,20260413T185911Z,unmatched_biospecimen_patient_uuid,patient_uuid,biospecimen_sample,bcr_patient_uuid,826d8224-55fe-48c7-a5fe-ad81ac1ebba3,biospecimen_sample contains this patient UUID ...,2,"Unmatched sample barcodes=[""TCGA-E2-A1IP-01A"",..."
4,20260413T185911Z,unmatched_biospecimen_patient_uuid,patient_uuid,biospecimen_sample,bcr_patient_uuid,e4321d99-ed3b-4fc8-b016-2ff47d93a064,biospecimen_sample contains this patient UUID ...,2,"Unmatched sample barcodes=[""TCGA-AN-A0FE-01A"",..."
5,20260413T185911Z,unmatched_source_case_submitter_id,case,source_metadata,case_submitter_id,TCGA-BH-A0B2,Source metadata includes this case submitter I...,1,Retained as an explicit audit mismatch note ra...


## Save review tables


In [4]:
cohort_preview_df = cohort_sorted_df.head(100).reset_index(drop=True)

cohort_preview_path = results_root / '61_minimal_dry_run_cohort_preview.tsv'
join_audit_review_path = results_root / '62_minimal_dry_run_join_audit.tsv'
row_counts_review_path = results_root / '63_minimal_dry_run_row_counts.tsv'
exclusions_review_path = results_root / '64_minimal_dry_run_exclusions.tsv'
ambiguity_flags_review_path = results_root / '65_minimal_dry_run_ambiguity_flags.tsv'
summary_review_path = results_root / '66_minimal_dry_run_summary.tsv'

cohort_preview_df.to_csv(cohort_preview_path, sep='\t', index=False)
join_review_df.to_csv(join_audit_review_path, sep='\t', index=False)
row_counts_review_df.to_csv(row_counts_review_path, sep='\t', index=False)
exclusions_review_df.to_csv(exclusions_review_path, sep='\t', index=False)
ambiguity_flags_review_df.to_csv(ambiguity_flags_review_path, sep='\t', index=False)
summary_review_df.to_csv(summary_review_path, sep='\t', index=False)

print(f'Saved: {cohort_preview_path}')
print(f'Saved: {join_audit_review_path}')
print(f'Saved: {row_counts_review_path}')
print(f'Saved: {exclusions_review_path}')
print(f'Saved: {ambiguity_flags_review_path}')
print(f'Saved: {summary_review_path}')

display(cohort_preview_df)
display(summary_review_df)


Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\61_minimal_dry_run_cohort_preview.tsv
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\62_minimal_dry_run_join_audit.tsv
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\63_minimal_dry_run_row_counts.tsv
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\64_minimal_dry_run_exclusions.tsv
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\65_minimal_dry_run_ambiguity_flags.tsv
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\66_minimal_dry_run_summary.tsv


,dry_run_build_id,provisional_patient_row_id,bcr_patient_barcode,bcr_patient_uuid,age_at_diagnosis,ajcc_metastasis_pathologic_pm,ajcc_nodes_pathologic_pn,ajcc_pathologic_tumor_stage,ajcc_staging_edition,ajcc_tumor_pathologic_pt,...,clinical_follow_up_v4_0__last_contact_days_to_json,clinical_follow_up_v4_0__new_tumor_event_dx_indicator_json,clinical_follow_up_v4_0__tumor_status_json,clinical_follow_up_v4_0__vital_status_json,biospecimen_sample__match_row_count,biospecimen_sample__bcr_patient_uuid_json,biospecimen_sample__bcr_sample_barcode_json,biospecimen_sample__bcr_sample_uuid_json,ambiguity_note_flags_json,join_status_flags_json
0,20260413T185911Z,1,TCGA-3C-AAAU,6E7D5EC6-A469-467C-B748-237353C23416,55,MX,NX,Stage X,6th,TX,...,"[""3928"", ""4047""]","[""YES"", ""NO""]","[""WITH TUMOR"", ""WITH TUMOR""]","[""Alive"", ""Alive""]",2,"[""6E7D5EC6-A469-467C-B748-237353C23416"", ""6E7D...","[""TCGA-3C-AAAU-01A"", ""TCGA-3C-AAAU-10A""]","[""197B76A1-A09A-4659-83AA-2C14FD1023A9"", ""E585...","[""parallel_patient_identifiers_retained"", ""cas...","[""followup_matched"", ""followup_multirow"", ""bio..."
1,20260413T185911Z,2,TCGA-3C-AALI,55262FCB-1B01-4480-B322-36570430C917,50,M0,N1a,Stage IIB,6th,T2,...,"[""4005""]","[""NO""]","[""TUMOR FREE""]","[""Alive""]",2,"[""55262FCB-1B01-4480-B322-36570430C917"", ""5526...","[""TCGA-3C-AALI-01A"", ""TCGA-3C-AALI-10A""]","[""0050D7C9-ECE9-4B6C-8023-1FF2EFCB3C9C"", ""8F55...","[""parallel_patient_identifiers_retained"", ""cas...","[""followup_matched"", ""biospecimen_sample_match..."
2,20260413T185911Z,3,TCGA-3C-AALJ,427D0648-3F77-4FFC-B52C-89855426D647,62,M0,N1a,Stage IIB,7th,T2,...,"[""1302"", ""1474""]","[""NO"", ""NO""]","[""TUMOR FREE"", ""TUMOR FREE""]","[""Alive"", ""Alive""]",2,"[""427D0648-3F77-4FFC-B52C-89855426D647"", ""427D...","[""TCGA-3C-AALJ-01A"", ""TCGA-3C-AALJ-10A""]","[""E355913A-6E4C-4D63-8A53-8BE4C5B003B8"", ""06BF...","[""parallel_patient_identifiers_retained"", ""cas...","[""followup_matched"", ""followup_multirow"", ""bio..."
3,20260413T185911Z,4,TCGA-3C-AALK,C31900A4-5DCD-4022-97AC-638E86E889E4,52,M0,N0 (i+),Stage IA,7th,T1c,...,"[""1221"", ""1448""]","[""NO"", ""NO""]","[""TUMOR FREE"", ""TUMOR FREE""]","[""Alive"", ""Alive""]",2,"[""C31900A4-5DCD-4022-97AC-638E86E889E4"", ""C319...","[""TCGA-3C-AALK-01A"", ""TCGA-3C-AALK-10A""]","[""2A532346-0008-4AF8-99BC-D709585DA1D6"", ""EBE9...","[""parallel_patient_identifiers_retained"", ""cas...","[""followup_matched"", ""followup_multirow"", ""bio..."
4,20260413T185911Z,5,TCGA-4H-AAAK,6623FC5E-00BE-4476-967A-CBD55F676EA6,50,M0,N2a,Stage IIIA,7th,T2,...,"[""348""]","[""NO""]","[""TUMOR FREE""]","[""Alive""]",2,"[""6623FC5E-00BE-4476-967A-CBD55F676EA6"", ""6623...","[""TCGA-4H-AAAK-01A"", ""TCGA-4H-AAAK-10A""]","[""685B5DF4-0620-40E2-AA09-0DD736BB5E2B"", ""3519...","[""parallel_patient_identifiers_retained"", ""cas...","[""followup_matched"", ""biospecimen_sample_match..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,20260413T185911Z,96,TCGA-A2-A1G0,2e538802-2a31-4774-89a8-c6e4381a887d,49,M0,N1a,Stage IIB,7th,T2,...,[],[],[],[],2,"[""2e538802-2a31-4774-89a8-c6e4381a887d"", ""2e53...","[""TCGA-A2-A1G0-01A"", ""TCGA-A2-A1G0-10A""]","[""3596c171-86d4-48ae-80a5-847c874d4860"", ""e1e5...","[""parallel_patient_identifiers_retained"", ""cas...","[""followup_unmatched"", ""biospecimen_sample_mat..."
96,20260413T185911Z,97,TCGA-A2-A1G1,75b5529b-565a-4eba-918a-910b2f441d32,85,M0,N1,Stage IIB,7th,T2,...,[],[],[],[],2,"[""75b5529b-565a-4eba-918a-910b2f441d32"", ""75b5...","[""TCGA-A2-A1G1-01A"", ""TCGA-A2-A1G1-10A""]","[""159debac-79e0-4727-940c-020e120f8347"", ""e09b...","[""parallel_patient_identifiers_retained"", ""cas...","[""followup_unmatched"", ""biospecimen_sample_mat..."
97,20260413T185911Z,98,TCGA-A2-A1G4,e3b555aa-7f0a-49c6-9b13-182c61a144c1,71,M0,N1a,Stage IIIA,7th,T3,...,[],[],[],[],2,"[""e3b555aa-7f0a-49c6-9b13-182c61a144c1"", ""e3b5...","[""TCGA-A2-A1G4-01A"", ""TCGA-A2-A1G4-10A""]"

,dry_run_build_id,summary_section,summary_metric,summary_value,notes
0,20260413T185911Z,ambiguity,carried_global_ambiguity_note_count,6,Global ambiguity-note flags written into every...
1,20260413T185911Z,ambiguity,upstream_ambiguity_readiness_signal,not_ready_pending_manual_review,The upstream ambiguity-resolution layer remain...
2,20260413T185911Z,ambiguity,upstream_blueprint_ambiguity_count,11,Current blueprint ambiguity count retained as ...
3,20260413T185911Z,design,biospecimen_policy,sample_anchor_only,biospecimen is represented only through groupe...
4,20260413T185911Z,design,endpoint_policy,candidate_columns_side_by_side_only,Patient and follow-up endpoint-like fields rem...
5,20260413T185911Z,design,proposed_unit_of_analysis,patient/case,The minimal dry-run cohort preserves patient/c...
6,20260413T185911Z,design,provisional_patient_universe,clinical_patient,clinical_patient remains the provisional patie...
7,20260413T185911Z,design,treatment_policy,excluded_from_row_build,One-to-many treatment-detail rows remain exclu...
8,20260413T185911Z,field_selection,baseline_field_count,27,Baseline fields selected from cohort_blueprint...
9,20260413T185911Z,field_selection,followup_endpoint_candidate_field_count,5,Follow-up endpoint candidate fields carried as...


This notebook remains review-only. It does not parse raw files, build a new cohort from raw sources, freeze the final endpoint, expand child biospecimen layers, or perform modeling.
